In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ─── Layout & data ──────────────────────────────────────────
ANALOGS = ['scl', 'scc', 'scs', 'scy', 'scr', 'scd']
LABELS  = {'scl': 'LEU', 'scc': 'CYS', 'scs': 'SER',
           'scy': 'TYR', 'scr': 'ARG$^+$', 'scd': 'ASP$^-$'}

FIRST_NS_DIR   = "../../not_avail/first_ns" #not available in this Github repository
FRAME_NUM_DIR  = "../../not_avail/frame_num" #not available in this Github repository
RAW_DATA_DIR   = "../data/distribution_data/raw_data"

DURATION_FIRST = 400   # ns
DURATION_RAW   = 600   # ns
WINDOW_NS      = 1     # smoothing window
FPS_RAW        = 100   # frames per ns for raw_data

Z_RANGES = [(0, 9.5), (9.5, 19.5), (19.5, 30.0), (30.0, 40.0)]
Z_LABELS = ['I: 0–9.5 Å', 'II: 9.5–19.5 Å',
            'III: 19.5–30 Å', 'IV: 30–40 Å']
Z_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
TRAJ_STYLES = {1: '-', 2: '--', 3: ':'}
TRAJS = [1, 2, 3]

# ─── Helpers ────────────────────────────────────────────────
def load_frame_num(analog, traj):
    """Return DataFrame with columns [section, n_frames] for first_ns data."""
    path = os.path.join(FRAME_NUM_DIR, analog, f"traj{traj}.dat")
    if not os.path.isfile(path):
        return None
    return pd.read_csv(path, sep=r'\s+', comment='#',
                       names=['section', 'n_frames'])

def counts_first_ns(analog, traj):
    """
    Count of residues per section for the first 400 ns.
    Uses the frame_num file to know how many frames belong to each section
    (section index == 1-based ns index, so section i covers ns [i-1, i]).
    Returns (time_ns, counts_matrix[n_windows, n_regions]).
    """
    contacts_path = os.path.join(FIRST_NS_DIR, analog,
                                  f"{analog}_contacts_{traj}.dat")
    if not os.path.isfile(contacts_path):
        return None, None
    fn = load_frame_num(analog, traj)
    if fn is None:
        return None, None

    df = pd.read_csv(contacts_path, sep=r'\s+', skiprows=1,
                     usecols=[0, 4], names=['frame', 'z'])
    df['abs_z'] = df['z'].abs()

    # Build a per-frame → section (ns) lookup from the frame_num file.
    n_frames_per_section = fn['n_frames'].to_numpy(dtype=int)
    section_ns = fn['section'].to_numpy(dtype=float)  # in ns (1-based)
    frame_to_section = np.repeat(section_ns, n_frames_per_section)
    total_frames_expected = int(n_frames_per_section.sum())

    # Guard against contacts files with more frames than declared.
    max_frame = int(df['frame'].max()) + 1
    n_use = min(max_frame, total_frames_expected)
    lookup_ns = np.empty(max_frame, dtype=float)
    lookup_ns[:n_use] = frame_to_section[:n_use]
    if max_frame > n_use:
        lookup_ns[n_use:] = frame_to_section[-1]

    df['time_ns'] = lookup_ns[df['frame'].to_numpy(dtype=int)]

    # Aggregate on WINDOW_NS windows: sum region counts then divide by
    # the total number of frames captured in that window.
    df['bin'] = (np.floor((df['time_ns'] - 0.5) / WINDOW_NS)
                 * WINDOW_NS + WINDOW_NS / 2.0)

    records = []
    for tb, grp in df.groupby('bin'):
        n_fr = grp['frame'].nunique()
        if n_fr == 0:
            continue
        row = {'time_ns': tb}
        for (zlo, zhi), lbl in zip(Z_RANGES, Z_LABELS):
            row[lbl] = ((grp['abs_z'] >= zlo) &
                        (grp['abs_z'] < zhi)).sum() / n_fr
        records.append(row)
    if not records:
        return None, None
    out = pd.DataFrame(records).sort_values('time_ns')
    return out['time_ns'].to_numpy(), out[Z_LABELS].to_numpy()

def counts_raw(analog, traj):
    """
    Count of residues per WINDOW_NS window for ns [400, 1000] using
    the fixed 100 fps of the raw_data files.
    """
    path = os.path.join(RAW_DATA_DIR, analog, f"{analog}_contacts_{traj}.dat")
    if not os.path.isfile(path):
        return None, None
    df = pd.read_csv(path, sep=r'\s+', comment='#',
                     usecols=[0, 4], names=['frame', 'z'], skiprows=1)
    df['abs_z']   = df['z'].abs()
    df['time_ns'] = DURATION_FIRST + df['frame'] / FPS_RAW
    df['bin']     = (np.floor((df['time_ns'] - DURATION_FIRST)
                              / WINDOW_NS) * WINDOW_NS
                     + WINDOW_NS / 2.0 + DURATION_FIRST)

    records = []
    for tb, grp in df.groupby('bin'):
        n_fr = grp['frame'].nunique()
        if n_fr == 0:
            continue
        row = {'time_ns': tb}
        for (zlo, zhi), lbl in zip(Z_RANGES, Z_LABELS):
            row[lbl] = ((grp['abs_z'] >= zlo) &
                        (grp['abs_z'] < zhi)).sum() / n_fr
        records.append(row)
    if not records:
        return None, None
    out = pd.DataFrame(records).sort_values('time_ns')
    return out['time_ns'].to_numpy(), out[Z_LABELS].to_numpy()

# ─── Plot ───────────────────────────────────────────────────
n_cols, n_rows = 2, 3
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(12, 2.2 * n_rows),
                          sharex=True, dpi=2000,
                          gridspec_kw={'hspace': 0.35, 'wspace': 0.20})
axes_flat = axes.flatten()

for idx, analog in enumerate(ANALOGS):
    ax = axes_flat[idx]

    for traj in TRAJS:
        parts_t, parts_c = [], []
        for loader in (counts_first_ns, counts_raw):
            t_arr, c_arr = loader(analog, traj)
            if t_arr is not None:
                parts_t.append(t_arr)
                parts_c.append(c_arr)
        if not parts_t:
            continue
        t_all = np.concatenate(parts_t)
        c_all = np.concatenate(parts_c, axis=0)
        order = np.argsort(t_all)
        t_all, c_all = t_all[order], c_all[order]

        for i, lbl in enumerate(Z_LABELS):
            ax.plot(t_all, c_all[:, i],
                    linestyle=TRAJ_STYLES[traj], color=Z_COLORS[i],
                    linewidth=0.9, alpha=0.85)

    ax.axvline(x=DURATION_FIRST, color='black', linestyle='--',
               lw=0.5, alpha=0.4)
    ax.set_xlim(0, 1000)
    ax.set_title(LABELS[analog], fontsize=10, fontweight='bold')
    ax.grid(True, which='major', linestyle='--', alpha=0.4, linewidth=0.3)
    ax.tick_params(axis='both', labelsize=8)

# Shared legend outside the subplots, at the top of the figure in one row.
region_lines = [Line2D([0], [0], color=Z_COLORS[i], lw=1.8) for i in range(4)]
traj_lines   = [Line2D([0], [0], color='grey',
                        linestyle=TRAJ_STYLES[t], lw=1.5) for t in TRAJS]
fig.legend(region_lines + traj_lines,
           Z_LABELS + [f'Traj {t}' for t in TRAJS],
           fontsize=9, ncol=len(Z_LABELS),  # regions on row 1, trajs on row 2
           loc='upper center', bbox_to_anchor=(0.5, 1.00),
           frameon=True, fancybox=False, edgecolor='black',
           borderpad=0.6, columnspacing=1.4, handlelength=2.2)

fig.text(0.5, 0.02, "Time (ns)", ha='center', fontsize=11)
fig.text(0.06, 0.5,
         f"Mean residue count per frame",
         va='center', rotation='vertical', fontsize=11)

os.makedirs("../plot", exist_ok=True)
out = "../plot/FigureS7.png"
plt.tight_layout(rect=[0.04, 0.03, 1, 0.91])
plt.savefig(out, dpi=600, bbox_inches='tight')
plt.show()
print("Saved →", out)